<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# **Procesamiento de Lenguaje Natural**
## **Desafio, Traductor**

### **Consigna**

* Replicar el modelo traductor desarrollado en clase (https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/blob/jul_2026/Clase%206/C%C3%B3digo/Traductor.ipynb) y extender su entrenamiento utilizando un conjunto de datos más amplio y secuencias de mayor longitud.
* Modificar valores de hiperparámetros (por ejemplo, el número de unidades en las capas LSTM) y analizar su impacto en el desempeño del traductor.
* Analizar el impacto del número de neuronas en las capas recurrentes, comparando el desempeño de distintas configuraciones del modelo.
* Generar y presentar al menos cinco ejemplos de traducciones producidas por el modelo entrenado.
* Interpretar a detalle los resultados obtenidos, considerando métricas de evaluación, calidad de las traducciones y posibles limitaciones del enfoque utilizado.

### **Actividades opcionales**

* Incorporar embeddings preentrenados para ambos idiomas y evaluar su efecto sobre el rendimiento del modelo.
* Experimentar con diferentes estrategias de generación de secuencias, como muestreo aleatorio (sampling) o búsqueda por haz (beam search).
* Implementar y entrenar una versión equivalente del modelo utilizando PyTorch, comparando los resultados con la implementación original.

## Resolución
Se plantea un dataset ampliado llevando los números máximos de pares de oraciones a 80000 y de vocabulario a 15000. Además, la máxima longitud de secuencia permitida es la máxima que se encontró en el dataset, es decir, no habría truncamiento.
Para comparar el impacto de los hiperparámetros se propone una "grilla" de entrenamiento, con distintas combinaciones, tal que cada modelo es xportado y guardado para su posterior evaluación de desempeño.
Finalmente, se realiza la evaluación de desempeño y generación de ejemplos; y se elige el mejor modelo resultante para modificarlo agregando una capa de embeddings preentrenados en el decoder.

In [9]:
import os
import json
import pickle
import time
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as pyplot
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Semillas
np.random.seed(40)
tf.random.set_seed(40)

MODELS_DIR = "models_traductor"
os.makedirs(MODELS_DIR, exist_ok=True)

MAX_NUM_SENTENCES = 80000
MAX_VOCAB_SIZE = 15000
max_input_len = 47
max_out_len   = 49

In [8]:
# Descarga y carga del dataset
url = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
zip_filename = "spa-eng.zip"

if not os.path.exists("spa-eng"):
    print("Descargando dataset")
    urllib.request.urlretrieve(url, zip_filename)
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Descarga y extracción completados")

with open("./spa-eng/spa.txt", "r", encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

print(f"Total oraciones: {len(lines)}")

np.random.shuffle(lines)

input_sentences, output_sentences, output_sentences_inputs = [], [], []

for i, line in enumerate(lines):
    if i >= MAX_NUM_SENTENCES:
        break
    if '\t' not in line:
        continue
    input_sentence, output = line.rstrip().split('\t')[:2]
    output_sentences.append(output + ' <eos>')
    output_sentences_inputs.append('<sos> ' + output)
    input_sentences.append(input_sentence)

print(f"Oraciones cargadas: {len(input_sentences)}")
print(f"Ejemplo EN (input): {input_sentences[5]}")
print(f"Ejemplo ES (output): {output_sentences[5]}")
print(f"Ejemplo ES (decoder input): {output_sentences_inputs[5]}")


Total oraciones: 118964
Oraciones cargadas: 80000
Ejemplo EN (input): Tom picked out an interesting book for Mary to read.
Ejemplo ES (output): Tom eligió un libro interesante para que María lea. <eos>
Ejemplo ES (decoder input): <sos> Tom eligió un libro interesante para que María lea.


In [13]:
# Tokenización y padding
# Tokenizador de entrada (Inglés)
input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

# Tokenizador de salida (Español)
output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(["<sos>", "<eos>"] + output_sentences)
output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
word2idx_outputs = output_tokenizer.word_index

num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)
num_words_inputs = min(len(word2idx_inputs) + 1, MAX_VOCAB_SIZE)

print(f"Vocabulario EN: {len(word2idx_inputs)} (usando top {num_words_inputs})")
print(f"Vocabulario ES: {len(word2idx_outputs)} (usando top {num_words_output})")
print(f"max_input_len: {max_input_len} | max_out_len: {max_out_len}")

# Padding
encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding="post")
decoder_output_sequences = pad_sequences(output_integer_seq, maxlen=max_out_len, padding="post")

# Se persisten los Tokenizers y metadata para reproducibilidad
with open(os.path.join(MODELS_DIR, 'input_tokenizer.pkl'), 'wb') as f:
    pickle.dump(input_tokenizer, f)

with open(os.path.join(MODELS_DIR, 'output_tokenizer.pkl'), 'wb') as f:
    pickle.dump(output_tokenizer, f)

dataset_metadata = {
    'MAX_NUM_SENTENCES': MAX_NUM_SENTENCES,
    'MAX_VOCAB_SIZE': MAX_VOCAB_SIZE,
    'max_input_len': max_input_len,
    'max_out_len': max_out_len,
    'num_words_output': num_words_output,
    'num_words_input': num_words_inputs
}

with open(os.path.join(MODELS_DIR, 'dataset_metadata.json'), 'w') as f:
    json.dump(dataset_metadata, f, indent=4)

print("Tokenizers y metadata guardados")

Vocabulario EN: 11871 (usando top 11872)
Vocabulario ES: 22220 (usando top 15000)
max_input_len: 47 | max_out_len: 49
Tokenizers y metadata guardados


In [16]:
# Pipeline de datos y split
def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)

    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,), dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Split (85/15)
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

enc_train, enc_val = encoder_input_sequences[:split_idx], encoder_input_sequences[split_idx:]
dec_in_train, dec_in_val = decoder_input_sequences[:split_idx], decoder_input_sequences[split_idx:]
dec_out_train, dec_out_val = decoder_output_sequences[:split_idx], decoder_output_sequences[split_idx:]
# La creación de los datasets se mueve al bucle de entrenamiento

print(f"Muestras de Entrenamiento: {len(enc_train)}")
print(f"Muestras de Validación:    {len(enc_val)}")

Muestras de Entrenamiento: 68000
Muestras de Validación:    12000


In [18]:
from spacy.ml import build_bow_text_classifier
# Carga de embeddings preentrenados de GloVe
def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID  = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

if not os.path.exists(_PKL_PATH) or not _is_valid_pickle(_PKL_PATH):
    print("Descargando gloveembedding.pkl")
    if os.path.exists(_PKL_PATH):
        os.remove(_PKL_PATH)
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=False)
    except Exception:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")
    if not _is_valid_pickle(_PKL_PATH):
        raise ValueError("El archivo descargado no es un pickle válido.")
    print("Descarga completada")
else:
    print("gloveembedding.pkl ya disponible")
    

def load_glove_embeddings(pkl_path):
    max_bytes = 2**28 - 1
    raw = bytearray()
    sz = os.path.getsize(pkl_path)
    with open(pkl_path, 'rb') as f:
        for _ in range(0, sz, max_bytes):
            raw += f.read(max_bytes)
    embeddings = pickle.loads(raw)
    idx_array  = np.arange(embeddings.shape[0])
    word2idx   = dict(zip(embeddings['word'], idx_array))
    return embeddings, word2idx

def get_word_embedding(word, embeddings, word2idx, n_features=50):
    i = word2idx.get(word, -1)
    return embeddings[i]['embedding'] if i != -1 else np.zeros(n_features)

def build_embedding_matrix(word2idx_inputs, embeddings, word2idx_glove, nb_words, embed_dim=50):
    matrix = np.zeros((nb_words, embed_dim))
    for word, i in word2idx_inputs.items():
        if i < nb_words:
            vec = get_word_embedding(word, embeddings, word2idx_glove, embed_dim)
            if vec is not None and len(vec) > 0:
                matrix[i] = vec
    return matrix

EMBED_DIM = 50
glove_embeddings, glove_word2idx = load_glove_embeddings(_PKL_PATH)
embedding_matrix_en = build_embedding_matrix(
    word2idx_inputs, glove_embeddings, glove_word2idx, num_words_inputs, EMBED_DIM
)
print(f"Matriz de embeddings EN creada con forma: {embedding_matrix_en.shape}")
print(f"Tokens sin embedding (ceros): {np.sum(np.sum(embedding_matrix_en**2, axis=1) == 0)}")

gloveembedding.pkl ya disponible
Matriz de embeddings EN creada con forma: (11872, 50)
Tokens sin embedding (ceros): 674


In [ ]:
# Se definen las funciones constructoras de Modelos Seq2Seq parametrizadas, tal que puedan
# soportar distintos hiperparámetros.

def build_encoder(nb_words, embed_dim, embedding_matrix, max_input_len, n_units, dropout_rate=0.3):
    enc_inputs = Input(shape=(max_input_len,), name='encoder_inputs')
    enc_emb_layer = Embedding(
        input_dim=nb_words,
        output_dim=embed_dim,
        weights=[embedding_matrix],
        trainable=False,
        name='encoder_embedding'
    )